# DiagnosticIQ pipeline example: test rules

An end-to-end walk through the DiagnosticIQ dataset-creation pipeline on the small hand-written test rule set (11 rules, 5 asset types). It runs in a few minutes on a CPU and covers every stage described in `docs/dataset-creation.md`:

1. convert the test rules into condition trees,
2. merge near-duplicate observations,
3. rank rules by rule-to-rule similarity (RRSim),
4. generate selection and elimination questions and format their text,
5. explore the generated questions,
6. compare them with the reference dataset, if the workspace has one.

The workspace is not part of this repository. Unpack the DiagnosticIQ test-rules workspace archive and point `ASSETOPSBENCH_DATASET_DIR` at it. It must contain:

```text
$ASSETOPSBENCH_DATASET_DIR/
├── test_datasets/rule_logic_test2.json   hand-written test rules
├── datasets/DiagnosticIQ/                DiagnosticIQ V3.1 (only its asset descriptions are used)
└── reference/                            optional: the original output, for step 6
```

Question options are sampled from Python sets of strings, whose iteration order depends on the per-process hash seed. Fix it before starting Jupyter so that reruns give the same questions:

```bash
export ASSETOPSBENCH_DATASET_DIR=/path/to/diagnosticiq_test_rules_workspace
PYTHONHASHSEED=0 uv run --group dataset-creation jupyter lab notebook/
```

Clear the outputs before committing this notebook: they contain rule and question text.

In [ ]:
import os
import random
from collections import Counter
from pathlib import Path

import numpy as np

import dataset_utils
from dataset_utils.paths import get_dataset_dir
from dataset_utils.utils import file_handle, grouping

dataset_dir = get_dataset_dir()
test_data_dir = dataset_dir / "test_datasets"
test_datasets_dir = test_data_dir / "datasets"
diagnosticiq_dir = dataset_dir / "datasets" / "DiagnosticIQ"
debug_dir = dataset_dir / "debug"
notebook_dir = Path(dataset_utils.__file__).resolve().parents[2] / "notebook"

required = [
    test_data_dir / "rule_logic_test2.json",
    diagnosticiq_dir / "data.jsonl",
    diagnosticiq_dir / "metadata.json",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing workspace inputs: {missing}")

if os.environ.get("PYTHONHASHSEED") in (None, "random"):
    print(
        "PYTHONHASHSEED is not fixed: correct answers are reproducible, "
        "but distractor options change from run to run."
    )
print("Workspace:", dataset_dir)

## Parameters

Change these and rerun from step 2 to see how each one shapes the dataset. The defaults are the configuration of the original test-rule dataset.

In [ ]:
DATASET_NAME = "DiagnosticIQTest"

# Observations whose pairwise similarity is above this quantile are merged.
OBSERVATION_MERGE_QUANTILE = 0.992

SEED = 420

CREATION_CONFIG = {
    "num_options": 4,
    "question_template_mode": "random",
    "question_template_random_count": 1,
    "sel_random_sample": 2,
    "elem_random_sample": 2,
    "n_least_sim_rules": 2,
}

## 1. Convert the test rules

Runs `diagnosticiq_source_test_rules_conversion.ipynb`, which parses each rule's condition list into an AND/OR tree and writes `test_datasets/TEST_rule_information.json` and `test_datasets/TEST_TreeStruct.pkl`. It prints every rule's tree.

In [ ]:
%run "{notebook_dir / 'diagnosticiq_source_test_rules_conversion.ipynb'}"

In [ ]:
rule_trees = file_handle.load_pickle(test_data_dir / "TEST_TreeStruct.pkl")
rule_information = file_handle.load_json(test_data_dir / "TEST_rule_information.json")

for rule in rule_information["rule_set"]:
    print(f"{rule['id']:<22} {rule['asset']:<15} {len(rule['display_text']['observations'])} observations")

## 2. Merge near-duplicate observations

Observation texts are embedded with `all-MiniLM-L6-v2`. Pairs above the similarity quantile are grouped, and each group is mapped to its longest observation, so that a question never offers two paraphrases of the same observation.

In [ ]:
from sentence_transformers import SentenceTransformer

# Built incrementally, as in the other notebooks: the set's iteration order
# feeds the grouping below.
unq_obs = set()
for d in rule_trees["rule_set"]:
    unq_obs = unq_obs.union(set(d["display_text"]["observations"]))
unq_obs = list(unq_obs)

model = SentenceTransformer("all-MiniLM-L6-v2")
enc = model.encode(unq_obs)
norm = np.outer(np.linalg.norm(enc, axis=1), np.linalg.norm(enc, axis=1)) + 1e-6
cosine_sim = np.abs((enc @ enc.T) / norm)

thresh = np.quantile(cosine_sim.flatten(), OBSERVATION_MERGE_QUANTILE)
cosine_sim = np.tril(cosine_sim, k=-1)
edges = list(zip(*np.where(cosine_sim > thresh)))
groups = grouping.getComponents(cosine_sim.shape[0], edges)

obs_mapping = {}
for group in groups:
    major = unq_obs[group[np.argmax([len(unq_obs[x]) for x in group])]]
    for x in group:
        obs_mapping[unq_obs[x]] = major

print(f"threshold {thresh:.4f}: {len(unq_obs)} observations -> {len(set(obs_mapping.values()))} after merging")
for group in groups:
    if len(group) > 1:
        print("\nmerged into:", obs_mapping[unq_obs[group[0]]])
        for x in group:
            print("   ", unq_obs[x])

## 3. Rule-to-rule similarity (RRSim)

Each rule's condition text is embedded with `all-mpnet-base-v2`. Distractors for a rule's questions are drawn from its least similar rules, so they are plausible for the asset but wrong for the conditions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

rule_repr_dict = {x["id"]: x["display_text"]["rules"] for x in rule_information["rule_set"]}
rule_ids = list(rule_repr_dict)

model = SentenceTransformer("all-mpnet-base-v2")
enc = model.encode(list(rule_repr_dict.values()))
norm = np.outer(np.linalg.norm(enc, axis=1), np.linalg.norm(enc, axis=1)) + 1e-6
rr_sim = np.abs((enc @ enc.T) / norm)

# Least similar first.
rule_id_similarity_map = {
    rule_id: [rule_ids[j] for j in np.argsort(rr_sim[i])] for i, rule_id in enumerate(rule_ids)
}

debug_dir.mkdir(exist_ok=True)
ax = sns.heatmap(rr_sim, xticklabels=rule_ids, yticklabels=rule_ids, cmap="viridis")
ax.set_title("RRSim: test rules")
plt.savefig(debug_dir / "RRSim_test_rules_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
RULE_TO_INSPECT = rule_ids[0]

i = rule_ids.index(RULE_TO_INSPECT)
print(f"Rules least similar to {RULE_TO_INSPECT} (distractor sources first):")
for other in rule_id_similarity_map[RULE_TO_INSPECT]:
    if other != RULE_TO_INSPECT:
        print(f"  {rr_sim[i, rule_ids.index(other)]:.3f}  {other}")

## 4. Generate and format the questions

`creation.create_dataset` expands each rule's condition tree into conjunctive trigger scenarios and writes *selection* questions (the correct option is one of the rule's observations) and *elimination* questions (the correct option comes from a dissimilar rule). `question_formatting` then writes the final question text.

In [ ]:
from dataset_utils import config, creation
from dataset_utils.outputs import question_formatting
from dataset_utils.reader import ADIQDataset

asset_descriptions = ADIQDataset(diagnosticiq_dir).asset_descriptions

dataset_config = config.DatasetCreationConfig(
    {
        **CREATION_CONFIG,
        "question_template_mode": config.QuestionTemplateMode(
            CREATION_CONFIG["question_template_mode"]
        ),
    }
)

random.seed(SEED)
np.random.seed(SEED)
dataset = creation.create_dataset(
    rule_trees,
    obs_mapping,
    asset_descriptions,
    rule_id_similarity_map,
    dataset_config,
)
creation.save_dataset(dataset, DATASET_NAME, test_datasets_dir)

ds = ADIQDataset(test_datasets_dir / DATASET_NAME)
for q in ds.questions:
    question_formatting(q, ds.asset_descriptions)
ds.save(DATASET_NAME, test_datasets_dir)

print(f"{len(ds.questions)} questions written to {test_datasets_dir / DATASET_NAME}")

## 5. Explore the questions

In [ ]:
questions_by_id = {q.question_id: q for q in ds.questions}

print("question types:", dict(Counter(q.question_type for q in ds.questions)))
print("options per question:", dict(Counter(len(q.options) for q in ds.questions)))
print("questions per rule:")
for rule_id, n in sorted(Counter(q.rule_name for q in ds.questions).items()):
    print(f"  {rule_id:<22} {n}")

In [ ]:
def show_question(q):
    print(f"{q.question_id}  rule {q.rule_name}  ({q.question_type})")
    print(q.question)
    for op_id, op, correct in zip(q.option_ids, q.options, q.correct):
        print(f"  {'*' if correct else ' '} {op_id}) {op}")


def questions_for_rule(rule_id, question_type=None):
    return [
        q
        for q in ds.questions
        if q.rule_name == rule_id and question_type in (None, q.question_type)
    ]


show_question(questions_for_rule(RULE_TO_INSPECT)[0])

In [ ]:
# Browse: change the index, the rule or the question type and rerun.
selected = questions_for_rule(rule_ids[1], question_type=None)
show_question(selected[0])

## 6. Compare with the reference dataset

If the workspace has `reference/`, compare the new dataset with the original output of this pipeline. The rule conversion, observation merging, rule similarity, question set and the correct answers of selection questions are deterministic and should match exactly. Options sampled from other rules depend on the hash seed (see the introduction): the distractors of selection questions and both the correct answer and the distractors of elimination questions. Those are only reported.

In [ ]:
reference_dir = dataset_dir / "reference" / "test_datasets"
reference_ds_dir = reference_dir / "datasets" / "simpleTestV1"


def without_dates(o):
    if isinstance(o, dict):
        return {k: without_dates(v) for k, v in o.items() if k != "date"}
    if isinstance(o, list):
        return [without_dates(x) for x in o]
    return o


def correct_options(q):
    return sorted(op for op, correct in zip(q["options"], q["correct"]) if correct)


if not reference_ds_dir.exists():
    print("No reference dataset in the workspace; skipping the comparison.")
else:
    new_meta = file_handle.load_json(test_datasets_dir / DATASET_NAME / "metadata.json")
    ref_meta = file_handle.load_json(reference_ds_dir / "metadata.json")
    checks = {
        "TEST_rule_information.json": without_dates(rule_information)
        == without_dates(file_handle.load_json(reference_dir / "TEST_rule_information.json")),
    }
    for key in ["creation_config", "asset_descriptions", "unique_observations", "rule_id_similarity_map", "rule_info"]:
        checks[f"metadata.{key}"] = without_dates(new_meta[key]) == without_dates(ref_meta[key])

    new_qs = {q["question_id"]: q for q in file_handle.load_jsonl(test_datasets_dir / DATASET_NAME / "data.jsonl")}
    ref_qs = {q["question_id"]: q for q in file_handle.load_jsonl(reference_ds_dir / "data.jsonl")}
    checks["question ids"] = new_qs.keys() == ref_qs.keys()
    shared = new_qs.keys() & ref_qs.keys()
    selection = [i for i in shared if ref_qs[i]["question_type"] == "positive"]
    elimination = [i for i in shared if ref_qs[i]["question_type"] != "positive"]
    checks["selection correct answers"] = all(
        correct_options(new_qs[i]) == correct_options(ref_qs[i]) for i in selection
    )

    for name, ok in checks.items():
        print(f"{'match' if ok else 'DIFFERS':<8} {name}")
    same_options = sum(Counter(new_qs[i]["options"]) == Counter(ref_qs[i]["options"]) for i in shared)
    same_elimination = sum(correct_options(new_qs[i]) == correct_options(ref_qs[i]) for i in elimination)
    print("\nHash-seed dependent:")
    print(f"  {same_elimination}/{len(elimination)} elimination questions have the same correct answer")
    print(f"  {same_options}/{len(shared)} questions have the same option set")